# Five common worry–harm clusters across World Risk Poll waves

We fit the 2021, 2023 and 2025 development populations independently with the same encoding. Each fit has seven weighted K-means centres, with the three centres for traffic, crime and severe weather mapped to the external-hazards cluster. The other four centres each define a cluster. This notebook replays the saved five-cluster evidence. The input packages contain aggregated answer-pattern counts and survey mass, fitted centres and predictions for those patterns. They contain no underlying respondent records. Run All checks assignments, resampling evidence, common rules and trends. An optional cell repeats the exhaustive rule search. No clustering fits are launched here.

In [1]:
from pathlib import Path
from itertools import product, combinations
import hashlib
import json
import tempfile

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.metrics import adjusted_rand_score
from threadpoolctl import threadpool_limits
from IPython.display import display

HERE=next((p for q in [Path.cwd(),*Path.cwd().parents] for p in [q,q/'paper',q/'github_publication/paper']
           if (p/'assets/shared_five_manifest.json').is_file()))
ASSETS=HERE/'assets'
manifest=json.loads((ASSETS/'shared_five_manifest.json').read_text())
for filename,expected in manifest['files_sha256'].items():
    assert hashlib.sha256((ASSETS/filename).read_bytes()).hexdigest()==expected

report=json.loads((ASSETS/'shared_five_report.json').read_text())
conditions=json.loads((ASSETS/'shared_five_rule_conditions.json').read_text())
RULE_NAMES=report['names']
YEARS=[2021,2023,2025]
with np.load(ASSETS/'shared_five_evidence.npz',allow_pickle=False) as pack:
    evidence={key:pack[key] for key in pack.files}
assert evidence['reference_counts'].sum()==90395

def geometry(answers):
    x=np.where(answers[:,:6]==2,1.,np.where(answers[:,:6]==1,1/3,0.))
    mean=x.mean(1,keepdims=True)
    return np.column_stack([(x-mean)*np.sqrt(.625/6),mean*np.sqrt(.625/6),
        answers[:,6:]*np.sqrt(.375/14),np.zeros(len(x))])

def predict(answers,year):
    centres=evidence[f'{year}_centres']
    groups=evidence[f'{year}_centre_to_cluster']
    return groups[cdist(geometry(answers),centres).argmin(1)]

def tab(x,y,weights):
    return np.bincount(5*x.astype('int64')+y,weights=weights,minlength=25).reshape(5,5)

def ari(t):
    def n2(v):return np.sum(v*(v-1)/2)
    n=t.sum()
    x,y=n2(t.sum(0)),n2(t.sum(1))
    e=x*y/(n*(n-1)/2)
    return (n2(t)-e)/((x+y)/2-e) if (x+y)/2!=e else 1.

reference=evidence['reference_answers']
counts=evidence['reference_counts']
mass=evidence['reference_mass']
for year in YEARS:
    for block in ['reference', 'native']:
        prefix='reference' if block=='reference' else f'native_{year}'
        key=f'reference_{year}_labels' if block=='reference' else f'native_{year}_labels'
        np.testing.assert_array_equal(predict(evidence[prefix+'_answers'],year),evidence[key])
print('Verified every saved point prediction against the fitted centres and group map.')

Verified every saved point prediction against the fitted centres and group map.


## Agreement on the same 2025 respondents

The counts give ordinary adjusted Rand index (ARI). Relative weights within a country follow WGT and every country has equal total weight. For a cluster, weighted Jaccard divides the weight of the intersection by that of the union under the two fits. Saved refit predictions are already aligned via their own wave's point fit to the 2025 labels. Use all 2,500 pairs for each wave pair and type of refit.

In [2]:
point_rows=[]
distributions=[]
cluster_rows=[]
for i,j in combinations(YEARS,2):
    x,y=[evidence[f'reference_{wave}_labels'] for wave in [i,j]]
    a=ari(tab(x,y,counts))
    agreement=np.trace(tab(x,y,mass))
    assert abs(a-adjusted_rand_score(np.repeat(x,counts),np.repeat(y,counts)))<1e-12
    point_rows.append(dict(year_a=i,year_b=j,ARI=a,weighted_agreement=agreement))
    for test in ['respondents','countries']:
        left=evidence[f'{i}_{test}_predictions']
        right=evidence[f'{j}_{test}_predictions']
        aris,jaccards=[],[]
        for x,y in product(left,right):
            aris.append(ari(tab(x,y,counts)))
            w=tab(x,y,mass)
            hit=np.diag(w)
            jaccards.append(hit/(w.sum(0)+w.sum(1)-hit))
        assert len(aris)==2500
        distributions.append(dict(year_a=i,year_b=j,test=test,p10=np.quantile(aris,.1),median=np.median(aris)))
        cluster_rows.extend(dict(year_a=i,year_b=j,test=test,cluster=c,p10=v)
                            for c,v in enumerate(np.quantile(jaccards,.1,axis=0)))

cross=pd.DataFrame(distributions)
jaccard=pd.DataFrame(cluster_rows).groupby('cluster').p10.min()
np.testing.assert_allclose(jaccard,report['recovery'],atol=1e-12)
np.testing.assert_allclose(cross[['p10','median']],
    pd.DataFrame(report['crossARI']).set_index(['year_a','year_b','test'])
    .loc[list(zip(cross.year_a,cross.year_b,cross.test)),['p10','median']],atol=1e-12)
display(pd.DataFrame(point_rows).round(4))
display(cross.round(4))
display(pd.DataFrame({'Cluster':RULE_NAMES,'Lowest cross-wave Jaccard p10':jaccard}).round(4))

,year_a,year_b,ARI,weighted_agreement
0,2021,2023,0.9652,0.9832
1,2021,2025,0.9371,0.9707
2,2023,2025,0.9374,0.9707


,year_a,year_b,test,p10,median
0,2021,2023,respondents,0.9129,0.9608
1,2021,2023,countries,0.9038,0.9350
2,2021,2025,respondents,0.9308,0.9409
3,2021,2025,countries,0.9219,0.9404
4,2023,2025,respondents,0.9295,0.9369
5,2023,2025,countries,0.9196,0.9273


,Cluster,Lowest cross-wave Jaccard p10
cluster,,
0,External hazards,0.8966
1,Basic necessities,0.8432
2,Mental-health worry,0.8560
3,Harm to people they know,0.8628
4,Few very high worries,0.9717


## One common rule for each cluster

The rule language uses AND and OR with at most three condition occurrences. Precision is the weighted share of rule matches in the target cluster. Coverage is the weighted share of cluster members matching its rule. Every selected rule reaches 80% precision in each wave. Among rules within one percentage point of maximum 2025 coverage, choose lowest cost, then highest 2025 precision and coverage, fewest original questions and canonical formula order.

The known-person-harm formula can show exactly four domains in its first branch because its other branch covers five or more. The few-very-high-worries rule allows food to be rated at any level and requires the other five domains to be below very worried. Its cost is three in the original search language.

In [3]:
def condition_masks(answers, conditions):
    counts = {'known': answers[:, 6:13].sum(1), 'personal': answers[:, 13:20].sum(1),
              'very': (answers[:, :6] == 2).sum(1), 'not': (answers[:, :6] == 0).sum(1),
              'some': (answers[:, :6] >= 1).sum(1)}
    operations = {'eq': np.equal, 'ge': np.greater_equal, 'le': np.less_equal}
    return np.column_stack([
        operations[c['op']](answers[:, c['j']] if 'j' in c else counts[c['base']], c['value'])
        for c in conditions])

def rule_clauses(kind, i, j=-1, k=-1):
    i, j, k = int(i), int(j), int(k)
    clauses = {0: [(i,)], 1: [(i, j)], 2: [(i,), (j,)],
               3: [(i, j, k)], 4: [(i,), (j,), (k,)],
               5: [(i,), (j, k)], 6: [(j,), (i, k)], 7: [(k,), (i, j)]}[int(kind)]
    return tuple(sorted(tuple(sorted(c)) for c in clauses))

def formula_string(clauses):
    return json.dumps({'clauses': clauses}, separators=(',', ':'))

def readable_rule(clauses, conditions):
    symbols = {'known': 'K', 'personal': 'P', 'very': 'V', 'not': 'N', 'some': 'A'}
    def atom(c):
        if c['kind'] == 'count_test':
            return symbols[c['base']] + (' >= ' if c['op'] == 'ge' else ' <= ') + str(c['value'])
        return c['label']
    parts = [' AND '.join(atom(conditions[i]) for i in clause) for clause in clauses]
    return ' OR '.join('(' + text + ')' if len(clauses)>1 and len(clause)>1 else text
                       for clause, text in zip(clauses, parts))

def formula_mask(clauses, tests):
    result = np.zeros(len(tests), dtype=bool)
    for clause in clauses:
        result |= np.all(tests[:, list(clause)], axis=1)
    return result

def predicate_intersections(population, assignment, n_clusters):
    tests, weights = population['tests'], population['weights']
    pattern_labels = np.full(len(weights), -1, dtype=int)
    pattern_labels[population['inverse']] = assignment
    assert np.array_equal(pattern_labels[population['inverse']], assignment)
    n = tests.shape[1]
    triples = np.array(list(combinations(range(n), 3)), dtype=np.int16)
    one = np.zeros((n, n_clusters))
    two = np.zeros((n, n, n_clusters))
    three = np.zeros((len(triples), n_clusters))
    with threadpool_limits(limits=4):
        for cluster in range(n_clusters):
            mask = pattern_labels == cluster
            block = tests[mask].astype(float)
            one[:, cluster] = block.T @ weights[mask]
            two[:, :, cluster] = block.T @ (block * weights[mask, None])
        offset = 0
        for i in range(n-2):
            j, k = np.triu_indices(n-i-1, k=1)
            for cluster in range(n_clusters):
                mask = tests[:, i] & (pattern_labels == cluster)
                block = tests[mask, i+1:].astype(float)
                joint = block.T @ (block * weights[mask, None])
                three[offset:offset+len(j), cluster] = joint[j, k]
            offset += len(j)
    assert offset == len(triples)
    return one, two, three, triples, np.bincount(pattern_labels, weights=weights, minlength=n_clusters)

def extract_description_rules(population, assignment, model, conditions,
                              candidate_file=None, epsilon=0.01, precision_floor=0.80):
    """Exhaustive DNF search, expanded cost <=3, with the agreed deterministic selection."""
    n = len(conditions)
    one, two, three, triples, totals = predicate_intersections(population, assignment, model['k'])
    pairs = np.array(list(combinations(range(n), 2)), dtype=np.int16)
    question_bits = np.array([sum(1 << q for q in set(c['questions'])) for c in conditions])
    bit_counts = np.array([i.bit_count() for i in range(1 << 13)])
    archives = []
    evaluated = 0

    def consider(kind, ids, intersections, cost):
        nonlocal evaluated
        # Roundoff in inclusion-exclusion can produce tiny values for empty events.
        intersections = np.where(np.abs(intersections)<1e-14, 0., intersections)
        intersections = np.minimum(np.maximum(intersections, 0.), totals)
        mass = intersections.sum(1)
        precision = np.divide(intersections, mass[:, None], out=np.zeros_like(intersections),
                              where=mass[:, None]>0)
        coverage = intersections / totals
        row, cluster = np.nonzero((mass[:, None]>0) & (precision>=precision_floor-1e-12))
        evaluated += len(ids)
        if not len(row):
            return
        selected = ids[row]
        q = bit_counts[np.bitwise_or.reduce(question_bits[selected], axis=1)]
        clauses = [rule_clauses(kind, *v) for v in selected]
        archives.append(pd.DataFrame(dict(cluster=cluster, kind=kind, i=selected[:,0],
            j=selected[:,1] if selected.shape[1]>1 else -1,
            k_atom=selected[:,2] if selected.shape[1]>2 else -1, cost=cost, questions=q,
            precision=precision[row,cluster], coverage=coverage[row,cluster],
            selected_mass=mass[row], true_mass=intersections[row,cluster],
            canonical_formula=[formula_string(c) for c in clauses])))

    consider(0, np.arange(n).reshape(-1,1), one.copy(), 1)
    a,b = pairs.T
    consider(1, pairs, two[a,b].copy(), 2)
    consider(2, pairs, one[a]+one[b]-two[a,b], 2)
    i,j,k = triples.T
    consider(3, triples, three.copy(), 3)
    consider(4, triples, one[i]+one[j]+one[k]-two[i,j]-two[i,k]-two[j,k]+three, 3)
    consider(5, triples, one[i]+two[j,k]-three, 3)
    consider(6, triples, one[j]+two[i,k]-three, 3)
    consider(7, triples, one[k]+two[i,j]-three, 3)
    assert evaluated == 1491084
    candidates = pd.concat(archives, ignore_index=True) if archives else pd.DataFrame(columns=['cluster'])
    if candidate_file is not None:
        candidates.to_csv(candidate_file, index=False, compression='gzip')
    selected_rules = []
    for cluster in range(model['k']):
        group = candidates.loc[candidates.cluster.eq(cluster)].copy()
        common = dict(wave=model['year'], trial=model['trial'], model_id=model['model_id'],
                      n_clusters=model['k'], cluster=cluster, formulas_evaluated=evaluated,
                      admissible_candidates=len(group), epsilon=epsilon, precision_floor=precision_floor)
        if group.empty:
            # A missing compact description does not remove an otherwise passing clustering.
            selected_rules.append(dict(common, status='no admissible rule under the declared budget'))
            continue
        maximum = group.coverage.max()
        band = group.loc[group.coverage>=maximum-epsilon-1e-12].copy()
        band['_p'] = band.precision.round(12)
        band['_r'] = band.coverage.round(12)
        best = band.sort_values(['cost','_p','_r','questions','canonical_formula'],
                               ascending=[True,False,False,True,True]).iloc[0]
        record = best.drop(labels=['_p','_r']).to_dict()
        clauses = rule_clauses(record['kind'],record['i'],record['j'],record['k_atom'])
        selected_rules.append(dict(record, **{key:value for key,value in common.items() if key!='cluster'},
                                   status='selected', best_coverage=maximum,
                                   rule=readable_rule(clauses, conditions)))
    return pd.DataFrame(selected_rules)

In [4]:
def rule_answers(model_answers):
    # Model inputs are worry, personal harm, known-person harm.
    # The original predicate IDs instead put known-person harm first.
    return model_answers[:,[*range(6),*range(13,20),*range(6,13)]]

def rule_measure(mask,target,mass,cluster):
    member=target==cluster
    hit=mass[mask&member].sum()
    selected=mass[mask].sum()
    total=mass[member].sum()
    return dict(precision=hit/selected if selected else np.nan,coverage=hit/total,
                selected_mass=selected,cluster_mass=total,intersection_mass=hit)

def score_rule_catalogue(candidates,answers,labels,mass,conditions):
    # Scores for every candidate that cleared 80% in 2025. The intersections
    # use the same inclusion–exclusion formulas as the full enumeration.
    population=dict(tests=condition_masks(rule_answers(answers),conditions),
                    weights=mass,inverse=np.arange(len(answers)))
    one,two,three,triples,totals=predicate_intersections(population,labels,5)
    n=len(conditions)
    lookup=np.full((n,n,n),-1,dtype='int32')
    lookup[tuple(triples.T)]=np.arange(len(triples))
    values=np.empty((len(candidates),2))
    for family,block in candidates.groupby('kind'):
        i,j,k=block[['i','j','k_atom']].to_numpy().T
        if family==0:
            intersections=one[i]
        elif family==1:
            intersections=two[i,j]
        elif family==2:
            intersections=one[i]+one[j]-two[i,j]
        else:
            positions=lookup[i,j,k]
            assert np.min(positions)>=0
            joint=three[positions]
            if family==3:
                intersections=joint
            elif family==4:
                intersections=one[i]+one[j]+one[k]-two[i,j]-two[i,k]-two[j,k]+joint
            elif family==5:
                intersections=one[i]+two[j,k]-joint
            elif family==6:
                intersections=one[j]+two[i,k]-joint
            else:
                assert family==7
                intersections=one[k]+two[i,j]-joint
        intersections=np.clip(np.where(abs(intersections)<1e-14,0.,intersections),0,totals)
        predicted_mass=intersections.sum(1)
        group=block.cluster.to_numpy()
        true=intersections[np.arange(len(block)),group]
        precision=np.divide(true,predicted_mass,out=np.zeros(len(block)),where=predicted_mass>0)
        values[block.index]=np.column_stack([precision,true/totals[group]])
    return values

def select_joint_rules(candidates,qualifying,conditions,criterion):
    selected=[]
    for c in range(5):
        pool=candidates.loc[qualifying & candidates.cluster.eq(c)]
        if pool.empty:
            selected.append(dict(criterion=criterion,cluster=c,name=RULE_NAMES[c],status='no admissible rule'))
            continue
        maximum=pool.coverage_2025.max()
        band=pool.loc[pool.coverage_2025.ge(maximum-.01-1e-12)].copy()
        band['_p']=band.precision_2025.round(12)
        band['_r']=band.coverage_2025.round(12)
        best=band.sort_values(['cost','_p','_r','questions','canonical_formula'],
                              ascending=[True,False,False,True,True]).iloc[0]
        formula=tuple(tuple(v) for v in json.loads(best.canonical_formula)['clauses'])
        selected.append(dict(criterion=criterion,cluster=c,name=RULE_NAMES[c],status='selected',
            canonical_formula=best.canonical_formula,rule=readable_rule(formula,conditions),
            cost=int(best.cost),questions=int(best.questions),eligible_candidates=len(pool),
            candidates_in_epsilon_band=len(band),best_coverage_2025=maximum))
    return pd.DataFrame(selected)

In [5]:
rows=[]
for y in YEARS:
    x=evidence[f'native_{y}_answers']
    labels=predict(x,y)
    w=evidence[f'native_{y}_mass']
    tests=condition_masks(rule_answers(x),conditions)
    for c,cl in enumerate(report['models'][f'full_{y}']['clusters']):
        rule=cl['rule']
        m=formula_mask(rule['canonical']['clauses'],tests)
        quality=rule_measure(m,labels,w,c)
        np.testing.assert_allclose([quality['precision'],quality['coverage']],
                                   [rule['precision'],rule['coverage']],atol=2e-12)
        assert quality['precision']>=.8 and rule['cost']<=3
        rows.append(dict(wave=y,cluster=c,name=cl['name'],rule=rule['text'],cost=rule['cost'],**quality))
results=pd.DataFrame(rows)
display(results[['wave','name','precision','coverage','cost']].round(4))

RUN_EXHAUSTIVE_SEARCH=False
if RUN_EXHAUSTIVE_SEARCH:
    x=evidence['native_2025_answers']
    pop=dict(tests=condition_masks(rule_answers(x), conditions),weights=evidence['native_2025_mass'],
             inverse=np.arange(len(x)))
    with tempfile.TemporaryDirectory() as t:
        path=Path(t)/'admissible.csv.gz'
        extract_description_rules(pop,predict(x,2025),dict(k=5,year=2025,trial='five',model_id='2025:five'),
                                  conditions,candidate_file=path)
        candidates=pd.read_csv(path,float_precision='round_trip').rename(columns={
            'precision':'precision_2025','coverage':'coverage_2025'})
    for y in [2021,2023]:
        x=evidence[f'native_{y}_answers']
        candidates[[f'precision_{y}',f'coverage_{y}']]=score_rule_catalogue(
            candidates,x,predict(x,y),evidence[f'native_{y}_mass'],conditions)
    good=candidates[[f'precision_{y}' for y in YEARS]].ge(.8-1e-12).all(axis=1)
    selected=select_joint_rules(candidates,good,conditions,'native')
    for r in selected.itertuples():
        assert json.loads(r.canonical_formula)==report['models']['full_2025']['clusters'][r.cluster]['rule']['canonical']
    print('All five common rules were reproduced by exhaustive search.')

,wave,name,precision,coverage,cost
0,2021,External hazards,0.8825,0.9096,3
1,2021,Basic necessities,0.8110,0.8536,3
2,2021,Mental-health worry,0.8612,0.9250,3
3,2021,Harm to people they know,0.8384,0.7410,3
4,2021,Few very high worries,0.9333,0.9874,3
5,2023,External hazards,0.8806,0.9017,3
6,2023,Basic necessities,0.8198,0.8363,3
7,2023,Mental-health worry,0.8159,0.9176,3
8,2023,Harm to people they know,0.8106,0.7340,3
9,2023,Few very high worries,0.9131,0.9813,3


## Changes in frequency using the same 2025 definitions

Apply the fixed 2025 model to each wave's own complete respondents. Average country shares across the same 92 development countries, and draw 1,000 paired country bootstrap samples for the change. This keeps the definition and country frame fixed as the survey wave changes.

In [6]:
shares=evidence['fixed_country_shares']
mean=shares.mean(axis=1)
np.testing.assert_allclose(mean,[report['shares'][str(y)] for y in YEARS],atol=1e-12)
display(pd.DataFrame(mean.T,index=RULE_NAMES,columns=YEARS).mul(100).round(2))
rng=np.random.default_rng(20261118)
indices=rng.integers(0,92,(1000,92))
changes=shares[2]-shares[0]
bounds=np.quantile(changes[indices].mean(1),[.025,.975],axis=0)
np.testing.assert_allclose(changes.mean(0),[r['delta'] for r in report['fixedChanges']],atol=1e-12)
np.testing.assert_allclose(bounds,[[r[k] for r in report['fixedChanges']] for k in ['lower','upper']],atol=1e-12)
display(pd.DataFrame({'Change':changes.mean(0),'Lower':bounds[0],'Upper':bounds[1]},index=RULE_NAMES).mul(100).round(2))

,2021,2023,2025
External hazards,28.65,30.02,30.44
Basic necessities,13.34,12.86,12.19
Mental-health worry,13.40,13.58,11.56
Harm to people they know,8.74,8.82,8.90
Few very high worries,35.87,34.71,36.92


,Change,Lower,Upper
External hazards,1.79,0.17,3.35
Basic necessities,-1.16,-2.07,-0.10
Mental-health worry,-1.83,-2.84,-0.91
Harm to people they know,0.15,-1.06,1.45
Few very high worries,1.05,-1.26,3.15


## Influence of encoded coordinates and original questions

We inspect the current frozen models on their native development populations. No centres or merge maps are fitted again. For each respondent, the mean is the average of that respondent's encoded worry answers. A relative worry coordinate is one answer minus this mean. Thus “water relative to mean” measures how much water worry stands above or below that person's other worries.

Three calculations answer different questions. For encoded coordinate $j$, weighted separation is $B_j=\sum_c W_c(\bar z_{cj}-\bar z_j)^2$; its share is $B_j/\sum_\ell B_\ell$. The feature scaling is the scaling used in the model. Coordinate omission drops the chosen squared-distance term for both respondents and centres, leaving all other terms fixed, and counts final-cluster changes. For original-question influence, replace an answer with an independent donor value from the same wave's weighted marginal, re-encode the complete answer vector, and count assignment changes. Enumerating every possible donor value gives the expectation exactly. Joint harm answers preserve the personal/known-person pair.

The mean and relative coordinates are derived from the same answers. Their omission sensitivities overlap and are not causal or uniquely attributable effects. The question calculation changes all affected derived coordinates coherently. These are the checks in Supplementary Section S2; the supplied manifest verifies the aggregate inputs and expected results.

In [7]:
from scipy.optimize import linear_sum_assignment
FI_DOMAINS=['food','water','crime','severe weather','traffic','mental health']
FI_WDOMAINS=['food','water','crime','severe weather','mental health']
FI_HARM=['food','water','crime','severe weather','traffic','mental health','work']
def fi_full_geometry(answers):
    v = np.array([0.0, 1 / 3, 1.0])[answers[:, :6]]
    m = v.mean(1, keepdims=True)
    return np.column_stack([(v - m) * np.sqrt(0.625 / 6), m * np.sqrt(0.625 / 6), answers[:, 6:] * np.sqrt(0.375 / 14), np.zeros(len(v))])

def fi_worry_geometry(answers, lam):
    v = np.array([0.0, 0.75, 1.0])[answers]
    m = v.mean(1, keepdims=True)
    return np.column_stack([v - m, np.sqrt(lam) * m]) / np.sqrt(5)

def fi_match_map(a, b, w, k):
    t = np.bincount(k * a.astype(int) + b, weights=w, minlength=k * k).reshape(k, k)
    i, j = linear_sum_assignment(-t)
    m = np.empty(k, dtype=np.int8)
    m[j] = i
    return m

def fi_means(X, labels, w, k):
    return np.asarray([np.average(X[labels == c], weights=w[labels == c], axis=0) for c in range(k)])

def fi_predict_model(model, answers=None, omit_parent=(), omit_child=(), mean_lambda=None):
    a = model['answers'] if answers is None else answers
    if model['family'] == 'full':
        X = fi_full_geometry(a)
        C = model['centres'].copy()
        if mean_lambda is not None:
            X[:, 6] *= np.sqrt(mean_lambda)
            C[:, 6] *= np.sqrt(mean_lambda)
        keep = np.setdiff1d(np.arange(X.shape[1]), omit_parent)
        return model['groups'][cdist(X[:, keep], C[:, keep], 'sqeuclidean').argmin(1)]
    X = fi_worry_geometry(a, 1.5)
    C = model['centres'].copy()
    if mean_lambda is not None:
        scale = np.sqrt(mean_lambda / 1.5)
        X[:, 5] *= scale
        C[:, 5] *= scale
    keep = np.setdiff1d(np.arange(X.shape[1]), omit_parent)
    parent = cdist(X[:, keep], C[:, keep], 'sqeuclidean').argmin(1)
    cut = parent == 2
    Z = fi_worry_geometry(a, 0.0)
    C2 = model['child_centres']
    keep2 = np.setdiff1d(np.arange(Z.shape[1]), omit_child)
    child = cdist(Z[:, keep2], C2[:, keep2], 'sqeuclidean').argmin(1)
    p = parent.astype(np.int8)
    p[cut] = 2 + child[cut]
    return model['groups'][p]

def fi_decomposition(X, labels, w, k, names, meta):
    w = w / w.sum()
    mu = fi_means(X, labels, w, k)
    overall = np.average(X, weights=w, axis=0)
    wc = np.bincount(labels, weights=w, minlength=k)
    by_cluster = wc[:, None] * (mu - overall) ** 2
    B = by_cluster.sum(0)
    T = (w[:, None] * (X - overall) ** 2).sum(0)
    residual = (w[:, None] * (X - mu[labels]) ** 2).sum(0)
    np.testing.assert_allclose(T, B + residual, atol=3e-15, rtol=3e-13)
    share = B / B.sum()
    eta = np.divide(B, T, out=np.zeros_like(B), where=T > 1e-25)
    rows = [dict(**meta, feature=name, TSS=float(T[j]), BSS=float(B[j]), WSS=float(residual[j]), separation_share=float(share[j]), eta_squared=float(eta[j])) for j, name in enumerate(names)]
    clusters = [dict(**meta, cluster=c, feature=name, BSS=float(by_cluster[c, j]), within_cluster_separation_share=float(by_cluster[c, j] / by_cluster[c].sum())) for c in range(k) for j, name in enumerate(names)]
    return (rows, clusters)

def fi_omission_list(family):
    if family == 'full':
        rows = [(t + ' (relative worry)', [j], [], 'coordinate') for j, t in enumerate(FI_DOMAINS)]
        rows += [('Overall worry mean', [6], [], 'coordinate')]
        rows += [('Personal harm: ' + t, [7 + j], [], 'coordinate') for j, t in enumerate(FI_HARM)]
        rows += [('Known-person harm: ' + t, [14 + j], [], 'coordinate') for j, t in enumerate(FI_HARM)]
        rows += [('All relative worries', list(range(6)), [], 'block'), ('All worry coordinates', list(range(7)), [], 'block'), ('All personal harm', list(range(7, 14)), [], 'block'), ('All known-person harm', list(range(14, 21)), [], 'block'), ('All harm', list(range(7, 21)), [], 'block')]
    else:
        rows = [(t + ' (relative worry, both stages)', [j], [j], 'coordinate') for j, t in enumerate(FI_WDOMAINS)]
        rows += [('Overall worry mean', [5], [5], 'coordinate'), ('All relative worries', list(range(5)), list(range(5)), 'block')]
        rows += [(t + ' (parent only)', [j], [], 'stage') for j, t in enumerate(FI_WDOMAINS)]
        rows += [(t + ' (child only)', [], [j], 'stage') for j, t in enumerate(FI_WDOMAINS)]
    return rows

def fi_randomisation_units(family):
    if family == 'worry':
        return [(t + ' worry', [j], 'original worry question') for j, t in enumerate(FI_WDOMAINS)]
    rows = [(t + ' worry', [j], 'original worry question') for j, t in enumerate(FI_DOMAINS)]
    rows += [('Personal harm: ' + t, [6 + j], 'personal indicator') for j, t in enumerate(FI_HARM)]
    rows += [('Known-person harm: ' + t, [13 + j], 'known-person indicator') for j, t in enumerate(FI_HARM)]
    rows += [('Harm question: ' + t, [6 + j, 13 + j], 'joint personal and known-person') for j, t in enumerate(FI_HARM)]
    return rows

def fi_inspect(models):
    pieces={name:[] for name in ['separation','cluster_separation','omissions','omission_clusters','questions','question_clusters']}
    for (family,y),m in models.items():
        X=m['X'];w=m['weights'];a=m['answers'];labels=m['labels'];k=m['k']
        np.testing.assert_array_equal(fi_predict_model(m),labels)
        if family=='full':
            names=[v+' (relative worry)' for v in FI_DOMAINS]+['Overall worry mean']
            names+=['Personal harm: '+v for v in FI_HARM]+['Known-person harm: '+v for v in FI_HARM]+['Unused zero coordinate']
            stages=[('final five clusters',X,labels,w,5,names)]
        else:
            names=[v+' (relative worry)' for v in FI_WDOMAINS]+['Overall worry mean']
            parent=m['parent_labels'];cut=parent==2
            child=cdist(m['child_X'],m['child_centres'],'sqeuclidean').argmin(1)
            stages=[('parent stage',X,parent,w,3,names),('child stage',m['child_X'][cut],child[cut],w[cut],2,names)]
        for stage,z,l,ww,n,names in stages:
            rows,clusters=fi_decomposition(z,l,ww,n,names,dict(family=family,wave=y,stage=stage))
            pieces['separation']+=rows;pieces['cluster_separation']+=clusters
        for name,parent,child,kind in fi_omission_list(family):
            pred=fi_predict_model(m,omit_parent=parent,omit_child=child)
            meta=dict(family=family,wave=y,feature=name,kind=kind)
            pieces['omissions'].append(dict(**meta,changed=float(w[pred!=labels].sum()/w.sum())))
            for c in range(k):
                old=labels==c;new=pred==c;hit=w[old&new].sum()
                pieces['omission_clusters'].append(dict(**meta,cluster=c,recovery=float(hit/w[old].sum()),Jaccard=float(hit/w[old|new].sum())))
        for name,cols,kind in fi_randomisation_units(family):
            states,inv=np.unique(a[:,cols],axis=0,return_inverse=True)
            probability=np.bincount(inv,weights=w,minlength=len(states))/w.sum()
            change=np.zeros(len(a))
            for state,p in zip(states,probability):
                modified=a.copy();modified[:,cols]=state
                change+=p*(fi_predict_model(m,answers=modified)!=labels)
            meta=dict(family=family,wave=y,question=name,kind=kind)
            pieces['questions'].append(dict(**meta,expected_changed=float(w@change/w.sum()),states=len(states)))
            for c in range(k):
                ok=labels==c
                pieces['question_clusters'].append(dict(**meta,cluster=c,expected_loss=float(w[ok]@change[ok]/w[ok].sum())))
    return {name:pd.DataFrame(rows) for name,rows in pieces.items()}

def fi_check_and_save(tables,family):
    keys=dict(separation=['family','wave','stage','feature'],cluster_separation=['family','wave','stage','cluster','feature'],
        omissions=['family','wave','kind','feature'],omission_clusters=['family','wave','kind','feature','cluster'],
        questions=['family','wave','kind','question'],question_clusters=['family','wave','kind','question','cluster'])
    numeric=dict(separation=['TSS','BSS','WSS','separation_share','eta_squared'],
        cluster_separation=['BSS','within_cluster_separation_share'],omissions=['changed'],
        omission_clusters=['recovery','Jaccard'],questions=['expected_changed','states'],question_clusters=['expected_loss'])
    folder=FI_ASSETS.parent/'results';folder.mkdir(exist_ok=True)
    for name,frame in tables.items():
        expected=pd.read_csv(FI_ASSETS/f'feature_importance_expected_{name}.csv',float_precision='round_trip')
        expected=expected[expected.family.eq(family)]
        left=frame.sort_values(keys[name]).reset_index(drop=True)
        right=expected.sort_values(keys[name]).reset_index(drop=True)
        pd.testing.assert_frame_equal(left[keys[name]],right[keys[name]],check_dtype=False)
        np.testing.assert_allclose(left[numeric[name]],right[numeric[name]],atol=3e-12,rtol=1e-10)
        frame.to_csv(folder/f'feature_importance_{family}_{name}.csv',index=False)
    print('Verified and saved',family,'feature-importance tables; no models refitted.')

In [8]:
FI_ASSETS=ASSETS
fi_manifest=json.loads((FI_ASSETS/'feature_importance_manifest.json').read_text())
for filename,expected in fi_manifest['files_sha256'].items():
    assert hashlib.sha256((FI_ASSETS/filename).read_bytes()).hexdigest()==expected
fi_models={}
for y in [2021,2023,2025]:
    a=evidence[f'native_{y}_answers'];X=fi_full_geometry(a)
    fi_models['full',y]=dict(family='full',wave=y,answers=a,weights=evidence[f'native_{y}_mass'],X=X,
        centres=evidence[f'{y}_centres'],groups=evidence[f'{y}_centre_to_cluster'],labels=evidence[f'native_{y}_labels'],k=5)
fi_tables=fi_inspect(fi_models)
fi_check_and_save(fi_tables,'full')
fi_s=fi_tables['separation'];fi_o=fi_tables['omissions']
fi_display=fi_s[fi_s.wave.eq(2025)].merge(fi_o[fi_o.wave.eq(2025)&fi_o.kind.eq('coordinate')],on=['family','wave','feature'])
fi_display['separation_pct']=100*fi_display.separation_share;fi_display['changed_pct']=100*fi_display.changed
display(fi_display[['feature','separation_pct','changed_pct']].sort_values('separation_pct',ascending=False).round(2))
display(fi_o[fi_o.feature.eq('Overall worry mean')][['wave','changed']].set_index('wave').mul(100).round(2))
display(fi_o[fi_o.wave.eq(2025)&fi_o.kind.eq('block')][['feature','changed']].set_index('feature').mul(100).round(2))
fi_q=fi_tables['questions']
display(fi_q[fi_q.kind.eq('original worry question')].pivot(index='question',columns='wave',values='expected_changed').mul(100).round(2))

Verified and saved full feature-importance tables; no models refitted.


,feature,separation_pct,changed_pct
6,Overall worry mean,20.87,10.86
5,mental health (relative worry),17.51,8.56
1,water (relative worry),14.80,8.35
4,traffic (relative worry),8.05,4.78
0,food (relative worry),7.79,5.01
2,crime (relative worry),4.03,4.07
18,Known-person harm: traffic,3.79,3.07
16,Known-person harm: crime,3.75,2.97
19,Known-person harm: mental health,3.35,2.98
20,Known-person harm: work,3.10,2.72


,changed
wave,
2021,9.82
2023,10.92
2025,10.86


,changed
feature,
All relative worries,35.73
All worry coordinates,53.91
All personal harm,2.09
All known-person harm,10.20
All harm,12.34


wave,2021,2023,2025
question,,,
crime worry,19.68,19.56,18.79
food worry,10.75,12.27,12.42
mental health worry,25.51,23.24,24.42
severe weather worry,19.22,20.42,17.29
traffic worry,22.88,22.63,23.24
water worry,24.14,23.56,22.50


The mean is the strongest single encoded coordinate in 2025: 20.87% of separation and 10.86% reassigned on omission. Mental-health and water deviations follow by omission. Known-person harm matters as a block: omitting its seven coordinates changes 10.20%, versus 2.09% for personal harm. The following check repeats mean omission on all 300 saved confirmation models, using their saved centres and fixed maps.

In [9]:
# Inspect every saved confirmation fit on the same native development population.
# Only fitted centres and their fixed maps are exported; no respondent rows or
# sampled respondent indices are needed by this check.
with np.load(FI_ASSETS/'feature_importance_refit_models.npz',allow_pickle=False) as pack:
    fi_refits={key:pack[key] for key in pack.files}
assert fi_refits['centres'].shape==(3,2,50,7,22)
fi_rows=[]
for yi,y in enumerate(fi_refits['years']):
    m=fi_models['full',int(y)];X=m['X'];w=m['weights'];keep=np.arange(22)!=6
    for ti,test in enumerate(fi_refits['tests']):
        for repeat in range(50):
            C=fi_refits['centres'][yi,ti,repeat];g=fi_refits['maps'][yi,ti,repeat]
            ref=g[cdist(fi_full_geometry(evidence['reference_answers']),C,'sqeuclidean').argmin(1)]
            np.testing.assert_array_equal(ref,evidence[f'{y}_{test}_predictions'][repeat])
            before=g[cdist(X,C,'sqeuclidean').argmin(1)]
            after=g[cdist(X[:,keep],C[:,keep],'sqeuclidean').argmin(1)]
            fi_rows.append(dict(family='full',wave=int(y),test=str(test),repeat=repeat,changed=float(w[before!=after].sum())))
fi_refit_table=pd.DataFrame(fi_rows)
fi_expected=pd.read_csv(FI_ASSETS/'feature_importance_expected_mean_refits.csv',float_precision='round_trip')
sort=['family','wave','test','repeat']
np.testing.assert_allclose(fi_refit_table.sort_values(sort).changed,fi_expected.sort_values(sort).changed,atol=3e-12)
fi_refit_table.to_csv(FI_ASSETS.parent/'results/feature_importance_full_mean_refits.csv',index=False)
display(fi_refit_table.groupby(['wave','test']).changed.agg(['min','median','max']).mul(100).round(2))

min  median    max
wave test                             
2021 countries     9.59   10.26  10.65
     respondents   9.70   10.19  10.56
2023 countries    10.55   10.81  11.04
     respondents  10.69   10.81  10.97
2025 countries    10.40   10.76  11.03
     respondents  10.51   10.82  11.01

## Complete primary fitting and selection

This section contains the executable primary fitting procedure, its original random-number schedules and all declared candidate grids. The native development population, missing-answer rules, equal-country weights, fitting budgets and reference answers remain fixed. Every wave fits its own centres. The code is included directly in this notebook; no source-code helper or course notebook is required.

Default execution loads aggregate centres and search results, recalculates the confirmation scores, and verifies the recorded shortlist. Point fits can also be repeated from the supplied global answer-pattern aggregates. Respondent and country refits require the original survey CSVs because the global aggregates do not retain respondents' countries. Obtain the recorded release, put the four `WRP_YEAR.csv` files in `paper/data/` or change `PRIMARY_RAW_FOLDER`. File checksums, question codes and parser settings are recorded in `primary_fitting_protocol.json`.

The functions below separate geometry, input preparation, sampling, fitting, scoring and selection. Both notebooks include this common implementation so that each remains self-contained. Optional fitting switches are all `False`. Recomputed results are written under `results/`; saved scientific evidence under `assets/` is preserved.

### Geometry and agreement calculations

Weighted k-means uses twenty initialisations and Lloyd iterations. Cross-wave scores compare the same reference answers; ARI uses respondent counts and Jaccard uses equal-country survey mass. The full model retains its stored zero coordinate for numerical compatibility. It contributes no distance.

In [10]:
"""Complete primary fitting functions; copied inline into both companions."""
from itertools import combinations, product
from pathlib import Path
import hashlib,json
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits

def pf_take(path):
    with np.load(path,allow_pickle=False) as z:return dict(z)

def pf_full_geometry(answers,setting):
    q=setting.get('q',1/3);alpha=setting.get('alpha',.625);beta=setting.get('beta',0.)
    x=np.array([0.,q,1.])[answers[:,:6]];mean=x.mean(1,keepdims=True)
    harm=answers[:,6:]*np.sqrt((1-alpha)/14)
    if setting.get('raw',False):return np.ascontiguousarray(np.column_stack([x*np.sqrt(alpha/6),harm]))
    joint=np.all(answers[:,[2,3]]==2,axis=1)*np.sqrt(alpha*beta)
    scale=np.sqrt(alpha*(1-beta)/6)
    return np.ascontiguousarray(np.column_stack([(x-mean)*scale,mean*scale,harm,joint]))

def pf_worry_geometry(answers,q=.75,mean=1.5,mode='ordinal'):
    if mode=='categorical':return np.eye(3)[answers].reshape(len(answers),-1)/np.sqrt(10)
    x=np.array([0.,q,1.])[answers]
    if mode=='raw':return np.ascontiguousarray(x/np.sqrt(5))
    m=x.mean(1,keepdims=True)
    return np.ascontiguousarray(np.column_stack([x-m,np.sqrt(mean)*m])/np.sqrt(5))

def pf_table(a,b,w,k):
    return np.bincount(k*a.astype(np.int64)+b,weights=w,minlength=k*k).reshape(k,k)

def pf_match(target,proposed,w,k):
    i,j=linear_sum_assignment(-pf_table(target,proposed,w,k))
    mapping=np.empty(k,dtype=np.int8);mapping[j]=i
    return mapping

def pf_ari(tables):
    t=np.asarray(tables,dtype=float)
    def pairs(x):return (x*(x-1)/2).sum(axis=-1)
    n=t.sum(axis=(-1,-2));den=n*(n-1)/2
    observed=pairs(t.reshape(*t.shape[:-2],-1))
    left=pairs(t.sum(-1));right=pairs(t.sum(-2));expected=left*right/den
    normal=(left+right)/2-expected
    return np.divide(observed-expected,normal,out=np.ones_like(normal),where=normal!=0)

def pf_jaccard(tables):
    hit=np.diagonal(tables,axis1=-2,axis2=-1);union=tables.sum(-1)+tables.sum(-2)-hit
    return np.divide(hit,union,out=np.zeros_like(hit),where=union>0)

def pf_pair_scores(left,right,counts,mass,k):
    # Aggregate patterns that have identical labels in every compared run.
    # This preserves all contingency tables while speeding up stable fits.
    within=right is None
    labels=np.asarray(left if within else np.concatenate([left,right]),dtype=np.int8)
    signature=np.ascontiguousarray(labels.T)
    keys=signature.view(np.dtype((np.void,signature.shape[1]))).ravel()
    _,first,inverse=np.unique(keys,return_index=True,return_inverse=True)
    labels=labels[:,first];cw=np.bincount(inverse,weights=counts);mw=np.bincount(inverse,weights=mass)
    indices=list(combinations(range(len(left)),2)) if within else list(product(range(len(left)),range(len(left),len(labels))))
    C=np.stack([pf_table(labels[i],labels[j],cw,k) for i,j in indices])
    M=np.stack([pf_table(labels[i],labels[j],mw,k) for i,j in indices])
    return pf_ari(C),pf_jaccard(M)

def pf_metrics(fits,data,k,repeats,anchor=None,groups=None,already_aligned=False):
    years=sorted(fits);reference=data[2025];rm=reference['ref_mass'];rc=reference['ref_counts']
    if anchor is None:anchor=fits[2025][0]['ref']
    to_anchor=np.arange(k,dtype=np.int8) if already_aligned else pf_match(anchor,fits[2025][0]['ref'],rm,k)
    if groups is None:groups=[[c] for c in range(k)]
    merge=np.empty(k,dtype=np.int8)
    for c,g in enumerate(groups):merge[g]=c
    out_k=len(groups);aligned={};shares=[];point=[];native=[];cross=[];clusters=[]
    for y in years:
        point_map=to_anchor if already_aligned else to_anchor[pf_match(fits[2025][0]['ref'],fits[y][0]['ref'],rm,k)]
        rows=[]
        for index,fit in enumerate(fits[y]):
            mapping=point_map if index==0 or already_aligned else point_map[pf_match(fits[y][0]['ref'],fit['ref'],rm,k)]
            rows.append({key:merge[mapping[fit[key]]] for key in ['native','ref']})
        aligned[y]=rows
        share=np.bincount(rows[0]['native'],weights=data[y]['mass'],minlength=out_k)
        shares.extend(dict(wave=y,cluster=c,share=float(v)) for c,v in enumerate(share))
    for a,b in combinations(years,2):
        C=pf_table(aligned[a][0]['ref'],aligned[b][0]['ref'],rc,out_k)
        M=pf_table(aligned[a][0]['ref'],aligned[b][0]['ref'],rm,out_k)
        point.append(dict(year_a=a,year_b=b,ARI=float(pf_ari(C)),Jaccard=float(pf_jaccard(M).min()),agreement=float(np.trace(M))))
    largest=max(s['share'] for s in shares)
    result=dict(k=out_k,largest_share=largest,dominance=out_k*largest,all_nonempty=all(s['share']>0 for s in shares),
        min_point_ARI=min(p['ARI'] for p in point),min_point_Jaccard=min(p['Jaccard'] for p in point))
    if repeats:
        for h,test in enumerate(['respondents','countries']):
            idx=slice(1+h*repeats,1+(h+1)*repeats)
            for y in years:
                values,_=pf_pair_scores(np.asarray([r['native'] for r in aligned[y][idx]]),None,data[y]['counts'],data[y]['mass'],out_k)
                native.append(dict(wave=y,test=test,p10=float(np.quantile(values,.1)),median=float(np.median(values))))
            for a,b in combinations(years,2):
                left=np.asarray([r['ref'] for r in aligned[a][idx]]);right=np.asarray([r['ref'] for r in aligned[b][idx]])
                values,j=pf_pair_scores(left,right,rc,rm,out_k)
                cross.append(dict(year_a=a,year_b=b,test=test,p10=float(np.quantile(values,.1)),median=float(np.median(values)),minimum=float(values.min())))
                clusters.extend(dict(year_a=a,year_b=b,test=test,cluster=c,p10=float(np.quantile(j[:,c],.1))) for c in range(out_k))
        result.update(native_min=min(min(r['p10'],r['median']) for r in native),
            min_cross_ARI=min(r['p10'] for r in cross),min_cross_Jaccard=min(r['p10'] for r in clusters))
        for name,cap,floor in [('strict',.4,.8),('sensitivity',.45,.78)]:
            if name=='sensitivity' and out_k==2:cap=.6
            structure=result['all_nonempty'] and largest<=cap and result['dominance']<3
            native_pass=structure and result['native_min']>=floor
            result[name+'_native']=native_pass
            result[name+'_pass']=native_pass and min(result['min_cross_ARI'],result['min_cross_Jaccard'])>=.8
    return result,dict(shares=shares,point=point,native=native,cross=cross,clusters=clusters)

### Survey preparation, original samples and independent fitting

The reader checks the survey checksum, restricts to the recorded development countries and substantive answers, and verifies the resulting aggregate counts and masses. Sampling reproduces the original 80% respondent and stratified-country draws. The parent and child fits use the same draw; the child seed is the parent seed XOR the recorded constant. No held-out countries enter fitting or selection.

In [11]:
def pf_aggregate_data(assets,family):
    if family=='full':
        e=pf_take(assets/'shared_five_evidence.npz')
        return {y:dict(answers=e[f'native_{y}_answers'],mass=e[f'native_{y}_mass'],counts=e[f'native_{y}_counts'],
            ref_answers=e['reference_answers'],ref_mass=e['reference_mass'],ref_counts=e['reference_counts']) for y in [2021,2023,2025]}
    e=pf_take(assets/'shared_worry_strict40_evidence.npz');r=pf_take(assets/'reference_common_worry.npz');saved=pf_take(assets/'primary_fitting_models.npz')
    return {y:dict(answers=e['answers'],mass=e['native_mass'][i],counts=saved['worry_native_counts'][i],
        ref_answers=e['answers'],ref_mass=e['worry_ref_mass'],ref_counts=r['counts'][3]) for i,y in enumerate([2019,2021,2023,2025])}

def pf_prepare_raw(assets,config,family,folder):
    """Read the pinned CSVs, freeze development eligibility and aggregate exactly."""
    result=pf_aggregate_data(assets,family)
    for y,d in result.items():
        source=config['csv_inputs'][str(y)];path=Path(folder)/source['name']
        assert hashlib.sha256(path.read_bytes()).hexdigest()==source['sha256'],f'Unexpected survey release: {path}'
        worry=['L6A','L6B','L6C','L6D','L6G'] if y==2019 else ['WP20720','WP20721','WP20722','WP20723']+(['WP22213'] if family=='full' else [])+['WP20726']
        harm=[f'WP224{j}' for j in range(42,49)] if family=='full' else []
        raw=pd.read_csv(path,usecols=['COUNTRY_ISO3','WGT',*worry,*harm],float_precision=source['float_precision'])
        registry=pd.read_csv(assets/f"country_split_{'full' if family=='full' else 'common_worry'}_{y}.csv").query("country_split == 'train'")
        keep=raw.COUNTRY_ISO3.isin(registry.COUNTRY_ISO3)&raw[worry].isin([1,2,3]).all(1)
        if harm:keep &= raw[harm].isin([1,2,3,4]).all(1)
        raw=raw.loc[keep];answers=(3-raw[worry].to_numpy()).astype(np.int8)
        if harm:answers=np.column_stack([answers,raw[harm].isin([1,3]),raw[harm].isin([2,3])]).astype(np.int8)
        patterns,inverse,counts=np.unique(answers,axis=0,return_inverse=True,return_counts=True)
        if family=='worry':
            inverse=answers@np.array([81,27,9,3,1]);patterns=d['answers'];counts=np.bincount(inverse,minlength=243)
        np.testing.assert_array_equal(patterns,d['answers']);np.testing.assert_array_equal(counts,d['counts'])
        codes=sorted(raw.COUNTRY_ISO3.unique());country=pd.Categorical(raw.COUNTRY_ISO3,categories=codes).codes
        wgt=raw.WGT.to_numpy();assert np.isfinite(wgt).all() and np.min(wgt)>0
        mass=np.bincount(inverse,weights=wgt/np.bincount(country,weights=wgt)[country]/len(codes),minlength=len(patterns))
        np.testing.assert_allclose(mass,d['mass'],atol=3e-13,rtol=0)
        reg=registry.set_index('COUNTRY_ISO3').loc[codes].reset_index()
        groups=[g.index.to_numpy() for _,g in reg.groupby(['GlobalRegion','CountryIncomeLevel'],sort=True)]
        sizes=np.array([len(g) for g in groups]);quota=np.maximum(1,np.floor(.8*sizes).astype(int));target=round(.8*len(codes))
        while quota.sum()<target:quota[np.where(quota<sizes,.8*sizes-quota,-np.inf).argmax()]+=1
        while quota.sum()>target:quota[np.where(quota>1,quota-.8*sizes,-np.inf).argmax()]-=1
        assert sum(map(len,groups))==len(codes)
        d.update(inverse=inverse,country=country,wgt=wgt,groups=groups,quota=quota,
            country_rows=[np.flatnonzero(country==i) for i in range(len(codes))],mass=mass)
    return result

def pf_mass(data,rows):
    country=data['country'][rows];w=data['wgt'][rows];total=np.bincount(country,weights=w)
    w=w/total[country]/np.count_nonzero(total)
    return np.bincount(data['inverse'][rows],weights=w,minlength=len(data['answers']))

def pf_draw(data,rng,test):
    if test=='respondents':
        return np.sort(np.concatenate([rng.choice(g,round(.8*len(g)),replace=False) for g in data['country_rows']]))
    chosen=np.concatenate([rng.choice(g,n,replace=False) for g,n in zip(data['groups'],data['quota'])])
    return np.flatnonzero(np.isin(data['country'],chosen))

def pf_full_masses(data,schedule,y):
    result=[]
    for task in schedule:
        if task['year']!=y:continue
        rows=np.arange(len(data['country'])) if task['test']=='point' else pf_draw(data,np.random.default_rng(task['sample_seed']),task['test'])
        result.append(pf_mass(data,rows))
    return np.asarray(result)

def pf_worry_masses(data,sample_seed,repeats):
    rng=np.random.default_rng(sample_seed);respondents=[];countries=[]
    for _ in range(repeats):
        respondents.append(pf_mass(data,pf_draw(data,rng,'respondents')))
        countries.append(pf_mass(data,pf_draw(data,rng,'countries')))
    return np.asarray([pf_mass(data,np.arange(len(data['country']))),*respondents,*countries])

def pf_kmeans(X,w,k,seed,tol):
    active=w>0
    with threadpool_limits(limits=1):
        model=KMeans(n_clusters=k,n_init=20,max_iter=300,tol=tol,algorithm='lloyd',random_state=int(seed))
        model.fit(X[active],sample_weight=w[active])
    assert model.n_iter_<300
    C=model.cluster_centers_;labels=cdist(X,C,'sqeuclidean').argmin(1).astype(np.int8)
    np.testing.assert_array_equal(labels[::29],model.predict(X[::29]))
    return C,labels

def pf_full_fits(data,setting,schedule,masses=None,point_only=False):
    result={}
    for y,d in data.items():
        tasks=[t for t in schedule if t['year']==y];weights=[d['mass']] if point_only else masses[y]
        X=pf_full_geometry(d['answers'],setting);R=pf_full_geometry(d['ref_answers'],setting);rows=[]
        for task,w in zip(tasks,weights):
            C,p=pf_kmeans(X,w,setting['k'],task['fit_seed'],1e-4)
            rows.append(dict(centres=C,native=p,ref=cdist(R,C,'sqeuclidean').argmin(1).astype(np.int8)))
        result[y]=rows
    return result

def pf_decode_full(data,setting,centres_by_year):
    return {y:[dict(centres=C,native=cdist(pf_full_geometry(d['answers'],setting),C,'sqeuclidean').argmin(1).astype(np.int8),
        ref=cdist(pf_full_geometry(d['ref_answers'],setting),C,'sqeuclidean').argmin(1).astype(np.int8)) for C in centres_by_year[y]] for y,d in data.items()}

def pf_worry_setting_geometry(answers,setting):
    mode=setting['mode']
    if mode=='categorical':return pf_worry_geometry(answers,mode=mode)
    return pf_worry_geometry(answers,setting['q'],float(mode.split('_')[1]) if mode.startswith('mean_') else 1.,'raw' if mode=='raw' else 'ordinal')

def pf_child_geometry(answers,setting):
    mode=setting['mode']
    if mode=='five':return pf_worry_geometry(answers,setting['q'],setting['mean'])
    if mode=='five_categorical':return pf_worry_geometry(answers,mode='categorical')
    cols={'crime_water':[1,2],'water':[1],'crime':[2]}[mode]
    return np.array([0.,setting['q'],1.])[answers[:,cols]]

def pf_worry_fits(data,setting,schedule,masses=None,point_only=False,split=False,parent_q=.75,parent_cache=None):
    result={}
    for y,d in data.items():
        seeds=schedule[str(y)]['fit_seeds'];weights=[d['mass']] if point_only else masses[y];a=d['answers']
        X=pf_worry_geometry(a,parent_q,1.5) if split else pf_worry_setting_geometry(a,setting)
        Z=pf_child_geometry(a,setting) if split else None
        target=np.where(a[:,2]==0,1,np.where(a[:,1]==0,0,2));rows=[]
        for index,(w,seed) in enumerate(zip(weights,seeds)):
            C,p=parent_cache[y][index] if parent_cache is not None else pf_kmeans(X,w,3 if split else setting['k'],seed,0.)
            if split:
                p=pf_match(target,p,d['ref_mass'],3)[p];cut=p==2
                child,cp=pf_kmeans(Z,w*cut,2,int(seed)^0xd5f489e7,0.)
                p=p.copy();p[cut]=2+cp[cut]
                rows.append(dict(parent_centres=C,child_centres=child,native=p,ref=p))
            else:rows.append(dict(centres=C,native=p,ref=p))
        result[y]=rows
    return result

def pf_decode_worry(data,setting,centres=None,parents=None,children=None,parent_q=.75):
    result={}
    for iy,(y,d) in enumerate(sorted(data.items())):
        a=d['answers'];rows=[]
        if centres is not None:
            X=pf_worry_setting_geometry(a,setting)
            for C in centres[iy]:
                p=cdist(X,C,'sqeuclidean').argmin(1).astype(np.int8);rows.append(dict(centres=C,native=p,ref=p))
        else:
            X=pf_worry_geometry(a,parent_q,1.5);Z=pf_child_geometry(a,setting)
            target=np.where(a[:,2]==0,1,np.where(a[:,1]==0,0,2))
            for C,child in zip(parents[iy],children[iy]):
                local=cdist(X,C,'sqeuclidean').argmin(1);p=pf_match(target,local,d['ref_mass'],3)[local]
                cut=p==2;p=p.copy();p[cut]=2+cdist(Z,child,'sqeuclidean').argmin(1)[cut]
                rows.append(dict(parent_centres=C,child_centres=child,native=p.astype(np.int8),ref=p.astype(np.int8)))
        result[y]=rows
    return result

def pf_shortlist_worry(frame):
    f=frame[frame.min_cross_Jaccard.notna()].copy();rank=['min_cross_Jaccard','min_cross_ARI','k','id'];asc=[False,False,False,True]
    pools=[f[f.strict_pass].sort_values(['k',*rank[:2],'id'],ascending=asc),
        f[f.sensitivity_pass&f.k.ge(3)].sort_values(rank,ascending=asc),f[f.sensitivity_pass&f.k.eq(2)].sort_values(rank,ascending=asc),
        f[f.strict_native].sort_values(rank,ascending=asc),f.sort_values(rank,ascending=asc)]
    chosen=[]
    for pool in pools:
        fresh=pool[~pool.id.isin(chosen)]
        if len(fresh) and len(chosen)<5:chosen.append(fresh.iloc[0].id)
    while len(chosen)<min(5,len(f)):
        chosen.append(pools[-1][~pools[-1].id.isin(chosen)].iloc[0].id)
    return chosen

def pf_shortlist_full(frame):
    f=frame[frame.refits_done].copy()
    def rank(row):return (-row.cross_Jaccard_p10_min,-row.cross_ARI_p10_min,-row.k,row.beta,row.setting)
    rows=list(f.itertuples());strict=sorted([r for r in rows if r.strict_all],key=lambda r:(-r.k,*rank(r)))
    pools=[strict,sorted([r for r in rows if r.strict_broad],key=rank),sorted([r for r in rows if r.sensitivity_broad],key=rank),
        sorted(rows,key=lambda r:(not r.sensitivity_native,-min(r.cross_ARI_p10_min,r.cross_Jaccard_p10_min),*rank(r)))]
    picked=[]
    def add(pool):
        found=next((r for r in pool if r.setting not in picked),None)
        if found is not None and len(picked)<3:picked.append(found.setting)
    for pool in pools[:3]:add(pool)
    if strict and len(picked)<3:add([r for r in pools[2] if r.sensitivity_all and r.k!=strict[0].k])
    while len(picked)<min(3,len(rows)):add(pools[-1])
    return picked

def pf_full_result(setting,stats,refits):
    r=dict(setting=setting['setting'],k=setting['k'],beta=setting['beta'],refits_done=refits,
        largest_native_share=stats['largest_share'],dominance=stats['dominance'],all_k_nonempty=stats['all_nonempty'],
        point_ARI_min=stats['min_point_ARI'],point_Jaccard_min=stats['min_point_Jaccard'],
        native_ARI_min=stats.get('native_min',np.nan),cross_ARI_p10_min=stats.get('min_cross_ARI',np.nan),cross_Jaccard_p10_min=stats.get('min_cross_Jaccard',np.nan))
    for name,cap,floor in [('strict',.4,.8),('sensitivity',.45,.78)]:
        structure=r['all_k_nonempty'] and r['largest_native_share']<=cap and r['dominance']<3
        native=structure and r['native_ARI_min']>=floor;broad=native and r['cross_ARI_p10_min']>=.8
        r.update({name+'_structure':structure,name+'_native':native,name+'_broad':broad,name+'_all':broad and r['cross_Jaccard_p10_min']>=.8})
    return r

def pf_check_stats(actual,expected,columns):
    for name in columns:
        np.testing.assert_allclose(actual[name],expected[name],atol=3e-12,rtol=1e-11,err_msg=name)

def pf_parent_cache(data,schedule,masses,parent_q=.75):
    return {y:[pf_kmeans(pf_worry_geometry(d['answers'],parent_q,1.5),w,3,seed,0.)
        for w,seed in zip(masses[y],schedule[str(y)]['fit_seeds'])] for y,d in data.items()}

### Pilot selection, fresh confirmation and optional sensitivity fitting

The complete grids and original seeds are supplied as JSON. The original size and stability filters, shortlist ordering and fresh confirmation schedules are implemented here. The final five-cluster full model applies its declared merge after fitting seven centres. For worry alone, the adopted `h005` split retains all five questions and the parent coding 0/0.75/1, with zero child mean weight. Several other passing settings give the same point assignments; the selected setting does not establish a unique clustering.

In [12]:
def pf_full_search(data,config,output):
    """Repeat the declared full-model pilot, shortlist and confirmation."""
    plan=config['full'];records={};bundles={}
    for stage,n in [('pilot',8),('confirmation',50)]:
        schedule=plan['schedule'][stage]
        masses={y:pf_full_masses(d,schedule,y) for y,d in data.items()}
        settings=plan['settings'] if stage=='pilot' else [s for s in plan['settings'] if s['setting'] in chosen]
        rows=[];bundles[stage]={}
        for s in settings:
            fits=pf_full_fits(data,s,schedule,point_only=True);stats,_=pf_metrics(fits,data,s['k'],0)
            allowed=stage=='confirmation' or (stats['all_nonempty'] and stats['largest_share']<=.45 and stats['dominance']<3)
            if allowed:
                fits=pf_full_fits(data,s,schedule,masses);stats,_=pf_metrics(fits,data,s['k'],n)
            rows.append(pf_full_result(s,stats,allowed));bundles[stage][s['setting']]=fits
        frame=pd.DataFrame(rows);records[stage]=frame
        frame.to_csv(output/f'recomputed_primary_full_{stage}.csv',index=False)
        if stage=='pilot':chosen=pf_shortlist_full(frame)
    source=bundles['confirmation']['k7_joint0'];anchor=pf_take(PF_ASSETS/'primary_fitting_models.npz')['full_anchor7']
    combined=[]
    for spec in plan['coarsenings']:
        stats,_=pf_metrics(source,data,7,50,anchor,spec['groups']);combined.append(dict(setting=spec['setting'],**stats))
    pd.DataFrame(combined).to_csv(output/'recomputed_primary_full_combinations.csv',index=False)
    return records,chosen,pd.DataFrame(combined),bundles

def pf_split_shortlist(frame):
    order=['strict_pass','min_cross_Jaccard','min_cross_ARI','id'];asc=[False,False,False,True]
    all_five=frame[frame['mode'].str.startswith('five')].sort_values(order,ascending=asc)
    targeted=frame[~frame['mode'].str.startswith('five')].sort_values(order,ascending=asc)
    return [*all_five.head(4).id,*targeted.head(3).id]

def pf_worry_search(data,config,which,output):
    """Repeat one declared worry-only grid, then fifty fresh confirmations."""
    plan=config['worry'][which];is_split=which=='split';bundles={};records={}
    for stage,n in [('pilot',8),('confirmation',50)]:
        schedule=plan['schedule'][stage]
        masses={y:pf_worry_masses(d,schedule[str(y)]['sample_seed'],n) for y,d in data.items()}
        parents=pf_parent_cache(data,schedule,masses) if is_split else None
        settings=plan['settings'] if stage=='pilot' else [s for s in plan['settings'] if s['id'] in chosen]
        rows=[];bundles[stage]={}
        for s in settings:
            fits=pf_worry_fits(data,s,schedule,point_only=True,split=is_split,
                parent_cache={y:p[:1] for y,p in parents.items()} if is_split else None)
            stats,_=pf_metrics(fits,data,s['k'],0)
            cap=.6 if s['k']==2 else .45
            allowed=(stage=='confirmation' or is_split or (stats['all_nonempty'] and stats['dominance']<3 and stats['largest_share']<=cap and stats['min_point_ARI']>=.62 and stats['min_point_Jaccard']>=.55))
            if allowed:
                fits=pf_worry_fits(data,s,schedule,masses,split=is_split,parent_cache=parents)
                stats,_=pf_metrics(fits,data,s['k'],n)
            rows.append(dict(**s,**{key:value for key,value in stats.items() if key!='k'}));bundles[stage][s['id']]=fits
        frame=pd.DataFrame(rows);records[stage]=frame
        frame.to_csv(output/f'recomputed_primary_worry_{which}_{stage}.csv',index=False)
        if stage=='pilot':chosen=pf_split_shortlist(frame) if is_split else pf_shortlist_worry(frame)
    return records,chosen,bundles

def pf_worry_confirmation(data,config,batch,output):
    """Refit both stages under the original or independent confirmation schedule."""
    plan=config['worry'][batch];schedule=plan['schedule']['confirmation']
    masses={y:pf_worry_masses(d,schedule[str(y)]['sample_seed'],50) for y,d in data.items()}
    setting=dict(id='h005',k=4,mode='five',q=.75,mean=0.)
    fits=pf_worry_fits(data,setting,schedule,masses,split=True)
    stats,details=pf_metrics(fits,data,4,50)
    pd.DataFrame([dict(batch=batch,**stats)]).to_csv(output/f'recomputed_primary_worry_{batch}.csv',index=False)
    return fits,stats,details

def pf_raw_worry_hierarchy(data,q,schedule,masses=None,point_only=False,controlled=False,parent_rank=2):
    """Raw five-question parent and child fits, independently within each wave."""
    result={};anchor_parent=None;anchor_selected=None
    for y in [2025,*[v for v in sorted(data) if v!=2025]]:
        d=data[y];X=pf_worry_geometry(d['answers'],q,mode='raw');rows=[]
        seeds=schedule[str(y)]['fit_seeds'];weights=[d['mass']] if point_only else masses[y]
        target=np.where(d['answers'][:,2]==0,1,np.where(d['answers'][:,1]==0,0,2))
        for index,(w,seed) in enumerate(zip(weights,seeds)):
            C,parent=pf_kmeans(X,w,3,seed,0.)
            if index==0:
                original_parent=parent.copy()
                if controlled:selected=int(np.flatnonzero(pf_match(target,parent,d['ref_mass'],3)==2)[0])
                elif y==2025:selected=sorted(range(3),key=lambda c:(float(C[c].mean()),tuple(C[c])))[parent_rank]
                else:selected=int(np.flatnonzero(pf_match(anchor_parent,parent,d['ref_mass'],3)==anchor_selected)[0])
                if y==2025:anchor_parent=parent.copy();anchor_selected=selected
                chosen=selected
            else:chosen=int(np.flatnonzero(pf_match(original_parent,parent,d['ref_mass'],3)==selected)[0])
            child,cp=pf_kmeans(X,w*(parent==chosen),2,int(seed)^0xd5f489e7,0.)
            others=sorted(c for c in range(3) if c!=chosen);labels=np.empty(len(parent),dtype=np.int8)
            for c,old in enumerate(others):labels[parent==old]=c
            labels[parent==chosen]=2+cp[parent==chosen]
            rows.append(dict(parent_centres=C,child_centres=child,split_parent=chosen,native=labels,ref=labels))
        result[y]=rows
    return result

def pf_raw_align(fits,data,k,point_maps=None,point_targets=None):
    """Freeze correspondence at the point fits; match refits to their own point."""
    rm=data[2025]['ref_mass'];result={}
    if point_maps is None:
        point_maps={y:pf_match(fits[2025][0]['ref'] if point_targets is None else point_targets[y],rows[0]['ref'],rm,k) for y,rows in fits.items()}
    for y,rows in fits.items():
        converted=[]
        for index,row in enumerate(rows):
            mapping=point_maps[y] if index==0 else point_maps[y][pf_match(rows[0]['ref'],row['ref'],rm,k)]
            converted.append(dict(native=mapping[row['native']],ref=mapping[row['ref']]))
        result[y]=converted
    return result

def pf_raw_comparison_refits(data,config,family,output):
    """Repeat the controlled and strongest confirmed raw comparisons."""
    if family=='full':
        schedule=config['full']['schedule']['confirmation']
        masses={y:pf_full_masses(d,schedule,y) for y,d in data.items()}
        evidence=pf_take(PF_ASSETS/'shared_five_evidence.npz');cases=[]
        for case,k,q in [('control',7,1/3),('best',3,.5)]:
            fits=pf_full_fits(data,dict(k=k,q=q,alpha=.625,raw=True),schedule,masses)
            if case=='control':
                point_maps={y:evidence[f'{y}_centre_to_cluster'][pf_match(cdist(pf_full_geometry(d['ref_answers'],dict(k=7)),evidence[f'{y}_centres'],'sqeuclidean').argmin(1),fits[y][0]['ref'],d['ref_mass'],7)] for y,d in data.items()}
                aligned=pf_raw_align(fits,data,7,point_maps=point_maps)
            else:aligned=pf_raw_align(fits,data,3)
            stats,_=pf_metrics(aligned,data,5 if case=='control' else 3,50,already_aligned=True)
            cases.append(dict(case=case,**stats))
    else:
        schedule=config['worry']['split']['schedule']['confirmation']
        masses={y:pf_worry_masses(d,schedule[str(y)]['sample_seed'],50) for y,d in data.items()};cases=[]
        for case,q in [('control',.75),('best',.875)]:
            fits=pf_raw_worry_hierarchy(data,q,schedule,masses,controlled=case=='control')
            targets={y:np.where((d['answers'][:,2]>0)&(d['answers'][:,1]>0)&(d['answers'][:,4]==0),3,np.where(d['answers'][:,2]==0,1,np.where(d['answers'][:,1]==0,0,2))) for y,d in data.items()} if case=='control' else None
            aligned=pf_raw_align(fits,data,4,point_targets=targets)
            stats,_=pf_metrics(aligned,data,4,50,already_aligned=True);cases.append(dict(case=case,**stats))
    frame=pd.DataFrame(cases);frame.to_csv(output/f'recomputed_centring_{family}.csv',index=False)
    return frame

def pf_raw_options(setting):
    k=setting['k'];identity=[[i] for i in range(k)]
    options=[dict(id=setting['id'],base_id=setting['id'],family=setting['family'],mode=setting['mode'],k=k,groups=identity,q=setting['q'])]
    if setting['family']=='full':
        options[0]['alpha']=setting['alpha']
        if k==7:
            for union in combinations(range(7),3):
                options.append(dict(id=setting['id']+'_merge'+''.join(map(str,union)),base_id=setting['id'],family='full',
                    mode='triple_merge',k=5,groups=[list(union)]+[[c] for c in range(7) if c not in union],q=setting['q'],alpha=setting['alpha']))
    return options

def pf_shortlist_raw(frame):
    order=['min_cross_Jaccard','min_cross_ARI','k','id'];asc=[False,False,False,True]
    pools=[frame[frame.strict_pass].sort_values(['k','min_cross_Jaccard','min_cross_ARI','id'],ascending=[False,False,False,True]),
        frame[(frame.native_min>=.8)&(frame.largest_share<=.4)&(frame.dominance<3)].sort_values(order,ascending=asc),
        frame[frame.sensitivity_pass].sort_values(order,ascending=asc),frame.sort_values(order,ascending=asc)]
    picked=[]
    for pool in pools:
        fresh=pool[~pool.id.isin(picked)]
        if len(fresh) and len(picked)<3:picked.append(fresh.iloc[0].id)
    while len(picked)<min(3,len(frame)):picked.append(pools[-1][~pools[-1].id.isin(picked)].iloc[0].id)
    return picked

def pf_raw_grid(data,config,family,settings,output):
    """Repeat the bounded raw-only screen, pilot, seal and fresh confirmation."""
    def fit_setting(s,stage,masses,point_only):
        if family=='full':return pf_full_fits(data,dict(s,raw=True),config['full']['schedule'][stage],masses,point_only)
        schedule=config['worry']['split']['schedule'][stage]
        if s['mode']=='hierarchy':return pf_raw_worry_hierarchy(data,s['q'],schedule,masses,point_only,parent_rank=s['parent_rank'])
        return pf_worry_fits(data,dict(s,mode='raw'),schedule,masses,point_only)
    all_settings=[s for s in settings if s['family']==family];pilot_runs={};point_rows=[];pilot_rows=[];lookup={}
    masses={}
    for stage,n in [('pilot',8),('confirmation',50)]:
        if family=='full':masses[stage]={y:pf_full_masses(d,config['full']['schedule'][stage],y) for y,d in data.items()}
        else:masses[stage]={y:pf_worry_masses(d,config['worry']['split']['schedule'][stage][str(y)]['sample_seed'],n) for y,d in data.items()}
    for s in all_settings:
        fits=fit_setting(s,'pilot',None,True);options=pf_raw_options(s);allowed=False
        for o in options:
            lookup[o['id']]=o
            stats,_=pf_metrics(fits,data,s['k'],0,groups=o['groups'])
            passed=stats['all_nonempty'] and stats['largest_share']<=.45 and stats['dominance']<3 and stats['min_point_ARI']>=.62 and stats['min_point_Jaccard']>=.5
            point_rows.append(dict(o,**{k:v for k,v in stats.items() if k!='k'},screen_pass=passed));allowed |= passed
        if not allowed:continue
        fits=fit_setting(s,'pilot',masses['pilot'],False);pilot_runs[s['id']]=fits
        for o in options:
            stats,_=pf_metrics(fits,data,s['k'],8,groups=o['groups'])
            pilot_rows.append(dict(o,**{k:v for k,v in stats.items() if k!='k'}))
    pilot=pd.DataFrame(pilot_rows);chosen=pf_shortlist_raw(pilot);rows=[]
    # No candidate or group union is added after this shortlist.
    selected=[lookup[name] for name in chosen]
    (output/f'recomputed_centring_{family}_shortlist.json').write_text(json.dumps(selected,indent=2)+'\n')
    for sid in sorted({o['base_id'] for o in selected}):
        setting=next(s for s in all_settings if s['id']==sid)
        fits=fit_setting(setting,'confirmation',masses['confirmation'],False)
        for o in selected:
            if o['base_id']!=sid:continue
            stats,_=pf_metrics(fits,data,setting['k'],50,anchor=pilot_runs[sid][2025][0]['ref'],groups=o['groups'])
            rows.append(dict(o,**{k:v for k,v in stats.items() if k!='k'}))
    for label,frame in [('point',pd.DataFrame(point_rows)),('pilot',pilot),('confirmation',pd.DataFrame(rows))]:
        frame.to_csv(output/f'recomputed_centring_{family}_{label}.csv',index=False)
    return chosen,pd.DataFrame(rows)

### Saved primary results and optional full-model fitting

The pilot has twenty count/interaction settings. Reconstruct the three-case confirmation shortlist from its pilot table, recalculate scores from every saved confirmation centre, then compare the two declared merges. The adopted five-cluster combination meets the original criteria. `RUN_PRIMARY_POINT_FITS` repeats the three independent point fits. `RUN_PRIMARY_REFITS` repeats their 100 refits per wave. `RUN_PRIMARY_SELECTION` reruns the declared pilot and confirmation grids.

In [13]:
PF_ASSETS=ASSETS
pf_manifest=json.loads((PF_ASSETS/'primary_fitting_manifest.json').read_text())
for filename,expected_hash in pf_manifest['files_sha256'].items():
    assert hashlib.sha256((PF_ASSETS/filename).read_bytes()).hexdigest()==expected_hash
PF_CONFIG=json.loads((PF_ASSETS/'primary_fitting_protocol.json').read_text())
pf_saved=pf_take(PF_ASSETS/'primary_fitting_models.npz')
pf_data=pf_aggregate_data(PF_ASSETS,'full')
PF_OUTPUT=PF_ASSETS.parent/'results';PF_OUTPUT.mkdir(exist_ok=True)

# The same rule that chose the confirmation shortlist is executable here.
pf_pilot=pd.read_csv(PF_ASSETS/'primary_full_pilot_summary.csv',float_precision='round_trip')
pf_selected=pf_shortlist_full(pf_pilot)
assert pf_selected==[s['setting'] for s in PF_CONFIG['full']['shortlist']]
display(pf_pilot[['setting','k','beta','refits_done','largest_native_share','native_ARI_min','cross_ARI_p10_min','cross_Jaccard_p10_min','strict_all']].round(2))
print('Sealed confirmation shortlist:',pf_selected)

# Recalculate confirmation quality from saved centres, not cached scores.
pf_confirmation=pd.read_csv(PF_ASSETS/'primary_full_confirmation_summary.csv',float_precision='round_trip').set_index('setting')
pf_checked=[];pf_full_runs={}
for setting in PF_CONFIG['full']['shortlist']:
    name=setting['setting']
    centres={y:pf_saved[f'full_confirmation_{name}_{y}'] for y in pf_data}
    fits=pf_decode_full(pf_data,setting,centres);pf_full_runs[name]=fits
    stats,_=pf_metrics(fits,pf_data,setting['k'],50)
    row=pf_full_result(setting,stats,True)
    pf_check_stats(row,pf_confirmation.loc[name],['largest_native_share','native_ARI_min','cross_ARI_p10_min','cross_Jaccard_p10_min'])
    pf_checked.append(row)
display(pd.DataFrame(pf_checked).round(2))

# The two declared groupings use the same seven-centre fits and fixed anchor.
pf_expected=pd.read_csv(PF_ASSETS/'primary_full_confirmation_coarsened_summary.csv',float_precision='round_trip').set_index('setting')
pf_combined=[]
for specification in PF_CONFIG['full']['coarsenings']:
    stats,_=pf_metrics(pf_full_runs['k7_joint0'],pf_data,7,50,pf_saved['full_anchor7'],specification['groups'])
    name=specification['setting'];row=dict(setting=name,**stats)
    expected=pf_expected.loc[name]
    pf_check_stats(dict(largest_native_share=stats['largest_share'],native_ARI_min=stats['native_min'],
        cross_ARI_p10_min=stats['min_cross_ARI'],cross_Jaccard_p10_min=stats['min_cross_Jaccard']),expected,
        ['largest_native_share','native_ARI_min','cross_ARI_p10_min','cross_Jaccard_p10_min'])
    pf_combined.append(row)
pf_combined=pd.DataFrame(pf_combined)
assert pf_combined.query("setting == 'five_combined'").strict_pass.item()
assert not pf_combined.query("setting == 'six_combined'").strict_pass.item()
display(pf_combined[['setting','k','largest_share','native_min','min_cross_ARI','min_cross_Jaccard','strict_pass']].round(2))
pf_combined.to_csv(PF_OUTPUT/'primary_full_combination_verification.csv',index=False)

def pf_primary_full_point_fits():
    setting=next(s for s in PF_CONFIG['full']['settings'] if s['setting']=='k7_joint0')
    return pf_full_fits(pf_data,setting,PF_CONFIG['full']['schedule']['confirmation'],point_only=True)

def pf_primary_full_refits(raw_folder):
    data=pf_prepare_raw(PF_ASSETS,PF_CONFIG,'full',raw_folder)
    setting=next(s for s in PF_CONFIG['full']['settings'] if s['setting']=='k7_joint0')
    schedule=PF_CONFIG['full']['schedule']['confirmation']
    masses={y:pf_full_masses(d,schedule,y) for y,d in data.items()}
    fits=pf_full_fits(data,setting,schedule,masses)
    stats,details=pf_metrics(fits,data,7,50,pf_saved['full_anchor7'],PF_CONFIG['full']['coarsenings'][1]['groups'])
    pd.DataFrame([stats]).to_csv(PF_OUTPUT/'recomputed_primary_full_confirmed.csv',index=False)
    for label,rows in details.items():pd.DataFrame(rows).to_csv(PF_OUTPUT/f'recomputed_primary_full_{label}.csv',index=False)
    return fits,stats

RUN_PRIMARY_POINT_FITS=False
RUN_PRIMARY_REFITS=False
RUN_PRIMARY_SELECTION=False
PRIMARY_RAW_FOLDER=PF_ASSETS.parent/'data'
if RUN_PRIMARY_POINT_FITS:
    pf_new_points=pf_primary_full_point_fits()
    print('Fitted three independent primary seven-centre point models.')
if RUN_PRIMARY_REFITS:
    pf_new_fits,pf_new_stats=pf_primary_full_refits(PRIMARY_RAW_FOLDER)
    display(pd.DataFrame([pf_new_stats]).round(2))
if RUN_PRIMARY_SELECTION:
    pf_raw=pf_prepare_raw(PF_ASSETS,PF_CONFIG,'full',PRIMARY_RAW_FOLDER)
    pf_new_search=pf_full_search(pf_raw,PF_CONFIG,PF_OUTPUT)
    print('Recomputed shortlist:',pf_new_search[1])
    display(pf_new_search[2].round(2))

# These recorded checks were produced by executing the point-fitting function
# above in the documented environment; default execution verifies saved results.
display(pd.read_csv(PF_ASSETS/'primary_point_fit_validation.csv').query("family == 'full'").round(2))

,setting,k,beta,refits_done,largest_native_share,native_ARI_min,cross_ARI_p10_min,cross_Jaccard_p10_min,strict_all
0,k3_joint0,3,0.00,False,0.49,NaN,NaN,NaN,False
1,k3_joint1,3,0.14,False,0.57,NaN,NaN,NaN,False
2,k3_joint2,3,0.25,False,0.52,NaN,NaN,NaN,False
3,k3_joint3,3,0.40,False,0.56,NaN,NaN,NaN,False
4,k4_joint0,4,0.00,True,0.43,0.61,0.47,0.06,False
5,k4_joint1,4,0.14,False,0.47,NaN,NaN,NaN,False
6,k4_joint2,4,0.25,False,0.46,NaN,NaN,NaN,False
7,k4_joint3,4,0.40,False,0.47,NaN,NaN,NaN,False
8,k5_joint0,5,0.00,True,0.41,0.51,0.49,0.00,False
9,k5_joint1,5,0.14,False,0.47,NaN,NaN,NaN,False


Sealed confirmation shortlist: ['k7_joint0', 'k6_joint1', 'k6_joint3']


,setting,k,beta,refits_done,largest_native_share,dominance,all_k_nonempty,point_ARI_min,point_Jaccard_min,native_ARI_min,cross_ARI_p10_min,cross_Jaccard_p10_min,strict_structure,strict_native,strict_broad,strict_all,sensitivity_structure,sensitivity_native,sensitivity_broad,sensitivity_all
0,k7_joint0,7,0.00,True,0.36,2.51,True,0.89,0.47,0.87,0.86,0.28,True,True,True,False,True,True,True,False
1,k6_joint1,6,0.14,True,0.41,2.49,True,0.76,0.28,0.68,0.71,0.26,False,False,False,False,True,False,False,False
2,k6_joint3,6,0.40,True,0.40,2.41,True,0.63,0.23,0.62,0.61,0.23,False,False,False,False,True,False,False,False


,setting,k,largest_share,native_min,min_cross_ARI,min_cross_Jaccard,strict_pass
0,six_combined,6,0.36,0.88,0.88,0.65,False
1,five_combined,5,0.36,0.91,0.90,0.84,True


,family,wave,fit_seed,stages,weighted_agreement,max_centre_difference
0,full,2021,2424861032,1,1.0,0.0
1,full,2023,1461228403,1,1.0,0.0
2,full,2025,1616620868,1,1.0,0.0


## Refitting without respondent-wise centring

For $d$ encoded worries, $\|x-y\|^2=\|(x-\bar x\mathbf1)-(y-\bar y\mathbf1)\|^2+d(\bar x-\bar y)^2$. Thus using raw scores increases the relative squared mean coefficient to $d$. The adopted coefficients are 1 for six-question worry–harm, 1.5 for the five-question worry-only parent and 0 for its child. Centring with the retained mean changes the distance geometry while preserving the original worry information in the full vector.

The controlled raw comparison keeps the original coding, balance and split/merge structure. Each wave independently fits raw coordinates, with no appended mean. The full raw point centres are matched to that wave's original seven centres before applying its frozen five-cluster map. The controlled worry-only model splits the raw parent matched to the original crime-and-water parent. The bounded raw reselection has its complete candidate list, criteria, pilot results and sealed shortlist supplied separately. The strongest confirmed full diagnostic has three clusters and still fails the original criteria. The selected raw four-cluster worry-only alternative passes, with different memberships.

The code below recomputes the table from saved aggregate centres and fixed maps. It preserves their correspondence, including poor matches. `RUN_RAW_SENSITIVITY_REFITS` repeats the two reported raw fits. `RUN_RAW_SELECTION` replays the complete bounded raw grid, screen, pilot, sealed shortlist and fresh confirmation. Both are off by default. These checks use development countries only and do not replace the adopted centred models.

In [14]:
PF_FAMILY='full'
def pf_decode_raw_case(data,family,case,stored):
    result={}
    for y,d in data.items():
        prefix=f'{family}_{case}_{y}';maps=stored[prefix+'_maps'];rows=[]
        if family=='full':
            setting=dict(raw=True,k=7 if case=='control' else 3,q=1/3 if case=='control' else .5,alpha=.625)
            X=pf_full_geometry(d['answers'],setting);R=pf_full_geometry(d['ref_answers'],setting)
            for C,m in zip(stored[prefix+'_centres'],maps):
                rows.append(dict(native=m[cdist(X,C,'sqeuclidean').argmin(1)],ref=m[cdist(R,C,'sqeuclidean').argmin(1)]))
        else:
            X=pf_worry_geometry(d['answers'],.75 if case=='control' else .875,mode='raw')
            for C,child,split,m in zip(stored[prefix+'_parents'],stored[prefix+'_children'],stored[prefix+'_split'],maps):
                parent=cdist(X,C,'sqeuclidean').argmin(1);others=sorted(c for c in range(3) if c!=split)
                local=np.empty(len(parent),dtype=np.int8)
                for c,old in enumerate(others):local[parent==old]=c
                cut=parent==split;local[cut]=2+cdist(X,child,'sqeuclidean').argmin(1)[cut]
                rows.append(dict(native=m[local],ref=m[local]))
        result[y]=rows
    return result

pf_raw_models=pf_take(PF_ASSETS/'centring_sensitivity_models.npz')
pf_raw_expected=pd.read_csv(PF_ASSETS/'centring_sensitivity_summary.csv',float_precision='round_trip')
pf_raw_rows=[];pf_raw_agreement=[]
for pf_case in ['control','best']:
    pf_raw_fits=pf_decode_raw_case(pf_data,PF_FAMILY,pf_case,pf_raw_models)
    pf_k=5 if PF_FAMILY=='full' and pf_case=='control' else 3 if PF_FAMILY=='full' else 4
    pf_raw_stats,_=pf_metrics(pf_raw_fits,pf_data,pf_k,50,already_aligned=True)
    pf_role='uncentred fixed-method diagnostic' if pf_case=='control' else 'best confirmed raw diagnostic' if PF_FAMILY=='full' else 'selected raw model'
    pf_record=pf_raw_expected[pf_raw_expected.family.eq(PF_FAMILY)&pf_raw_expected.role.eq(pf_role)].iloc[0]
    pf_check_stats(pf_raw_stats,pf_record,['largest_share','native_min','min_cross_ARI','min_cross_Jaccard'])
    pf_raw_rows.append(dict(case=pf_case,**pf_raw_stats))
    for y,d in pf_data.items():
        if PF_FAMILY=='full':
            old=pf_take(PF_ASSETS/'shared_five_evidence.npz')[f'native_{y}_labels'];pad=5
        else:
            a=d['answers'];p=np.where(a[:,2]==0,1,np.where(a[:,1]==0,0,2));old=np.where((p==2)&(a[:,4]==0),3,p);pad=4
        score=float(pf_ari(pf_table(old,pf_raw_fits[y][0]['native'],d['counts'],pad)))
        pf_raw_agreement.append(dict(family=PF_FAMILY,wave=y,case=pf_case,ARI=score))
display(pd.DataFrame(pf_raw_rows).round(2))
pd.DataFrame(pf_raw_rows).to_csv(PF_OUTPUT/f'centring_{PF_FAMILY}_verification.csv',index=False)
pf_agreement=pd.DataFrame(pf_raw_agreement)
pf_agreement_expected=pd.read_csv(PF_ASSETS/'centring_expected_model_agreement.csv',float_precision='round_trip')
for row in pf_agreement.itertuples():
    expected=pf_agreement_expected[pf_agreement_expected.family.eq(row.family)&pf_agreement_expected.wave.eq(row.wave)&pf_agreement_expected['case'].eq(row.case)].ARI.item()
    np.testing.assert_allclose(row.ARI,expected,atol=3e-12)
display(pf_agreement.pivot(index='case',columns='wave',values='ARI').round(2))
pf_agreement.to_csv(PF_OUTPUT/f'centring_{PF_FAMILY}_agreement.csv',index=False)

# These complete grids, thresholds and pilot/confirmation records document the
# bounded raw reselection. No new search is run by the default replay.
pf_raw_settings=json.loads((PF_ASSETS/'centring_candidate_settings.json').read_text())
display(pd.DataFrame([s for s in pf_raw_settings if s['family']==PF_FAMILY]))
pf_raw_pilot=pd.read_csv(PF_ASSETS/'centring_pilot_candidates.csv',float_precision='round_trip')
pf_raw_confirm=pd.read_csv(PF_ASSETS/'centring_confirmation_candidates.csv',float_precision='round_trip')
display(pf_raw_confirm[pf_raw_confirm.family.eq(PF_FAMILY)][['id','k','largest_share','native_min','min_cross_ARI','min_cross_Jaccard','strict_pass']].round(2))

RUN_RAW_SENSITIVITY_REFITS=False
if RUN_RAW_SENSITIVITY_REFITS:
    pf_raw_data=pf_prepare_raw(PF_ASSETS,PF_CONFIG,PF_FAMILY,PRIMARY_RAW_FOLDER)
    display(pf_raw_comparison_refits(pf_raw_data,PF_CONFIG,PF_FAMILY,PF_OUTPUT).round(2))


pf_raw_chosen=pf_shortlist_raw(pf_raw_pilot[pf_raw_pilot.family.eq(PF_FAMILY)])
pf_raw_sealed=json.loads((PF_ASSETS/'centring_confirmation_shortlist.json').read_text())
assert pf_raw_chosen==[s['id'] for s in pf_raw_sealed if s['family']==PF_FAMILY]
print('Verified bounded raw-search shortlist:',pf_raw_chosen)
RUN_RAW_SELECTION=False
if RUN_RAW_SELECTION:
    pf_raw_data=pf_prepare_raw(PF_ASSETS,PF_CONFIG,PF_FAMILY,PRIMARY_RAW_FOLDER)
    pf_raw_rerun=pf_raw_grid(pf_raw_data,PF_CONFIG,PF_FAMILY,pf_raw_settings,PF_OUTPUT)
    print('Recomputed raw shortlist:',pf_raw_rerun[0])
    display(pf_raw_rerun[1].round(2))
display(pd.read_csv(PF_ASSETS/'centring_point_fit_validation.csv').query('family == @PF_FAMILY').round(2))

,case,k,largest_share,dominance,all_nonempty,min_point_ARI,min_point_Jaccard,native_min,min_cross_ARI,min_cross_Jaccard,strict_native,strict_pass,sensitivity_native,sensitivity_pass
0,control,5,0.40,2.01,True,0.55,0.0,0.53,0.52,0.00,False,False,False,False
1,best,3,0.41,1.22,True,0.77,0.8,0.93,0.72,0.76,False,False,True,False


wave,2021,2023,2025
case,,,
best,0.36,0.35,0.34
control,0.53,0.71,0.72


,alpha,family,id,k,mode,q
0,0.500,full,F3_q0.250000_a0.500,3,flat,0.250000
1,0.625,full,F3_q0.250000_a0.625,3,flat,0.250000
2,0.750,full,F3_q0.250000_a0.750,3,flat,0.250000
3,0.500,full,F3_q0.333333_a0.500,3,flat,0.333333
4,0.625,full,F3_q0.333333_a0.625,3,flat,0.333333
5,0.750,full,F3_q0.333333_a0.750,3,flat,0.333333
6,0.500,full,F3_q0.500000_a0.500,3,flat,0.500000
7,0.625,full,F3_q0.500000_a0.625,3,flat,0.500000
8,0.750,full,F3_q0.500000_a0.750,3,flat,0.500000
9,0.500,full,F4_q0.250000_a0.500,4,flat,0.250000


,id,k,largest_share,native_min,min_cross_ARI,min_cross_Jaccard,strict_pass
0,F3_q0.500000_a0.500,3,0.40,0.85,0.68,0.72,False
1,F3_q0.500000_a0.625,3,0.41,0.93,0.72,0.76,False
2,F3_q0.500000_a0.750,3,0.41,0.88,0.68,0.72,False


Verified bounded raw-search shortlist: ['F3_q0.500000_a0.500', 'F3_q0.500000_a0.625', 'F3_q0.500000_a0.750']


,family,case,wave,weighted_agreement,maximum_centre_difference
0,full,control,2021,1.0,0.0
1,full,control,2023,1.0,0.0
2,full,control,2025,1.0,0.0
3,full,best,2021,1.0,0.0
4,full,best,2023,1.0,0.0
5,full,best,2025,1.0,0.0


The point-fitting checks reproduce all seven adopted primary models exactly in membership. Input decoding and sampling checks also reproduce 2,242 original native/resampling mass vectors. The verification records below are aggregate summaries; they contain no respondent rows or sampled indices.

In [15]:
pf_sampling=pd.read_csv(PF_ASSETS/'primary_sampling_validation.csv')
display(pf_sampling[pf_sampling.family.eq(PF_FAMILY)].groupby(['study','stage']).agg(draws=('draws','sum'),maximum_mass_difference=('maximum_mass_difference','max')))

draws  maximum_mass_difference
study   stage                                       
primary confirmation    303                      0.0
        pilot            51                      0.0